# Working with Databases: SQLite vs DuckDB

## Introduction

In earlier projects you loaded data from CSV files and MongoDB. Today we step into **relational databases** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â structured data stores that enforce relationships between tables, guarantee data consistency, and answer complex analytical questions orders of magnitude faster than reading flat files.

> ÃƒÂ¢Ã‚ÂÃ¢â‚¬Å“ **Why does this matter for the Nepal earthquake project?** The 2015 Gorkha earthquake damage survey produced four interrelated tables: building structure, building damage, household demographics, and a cross-reference map. Every meaningful question ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â "Which districts suffered the worst damage?" "Does building age predict severe damage?" "How does caste correlate with damage outcomes?" ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â requires combining multiple tables. That combination is exactly what relational databases were designed for.

Our two tools for this lesson:
- **SQLite** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a row-based relational database embedded in a single file (`nepal.sqlite`), ideal for transactional workloads and embedded applications. Think: the database behind your mobile banking app.
- **DuckDB** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a column-based analytical engine that can query CSV files *directly*, no import step needed. Think: a spreadsheet engine that speaks fluent SQL and is 10ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“100ÃƒÆ’Ã¢â‚¬â€ faster for the analytics data scientists run.

Both tools share the same SQL syntax you already know. The difference is in *architecture* ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â and architecture determines speed.

## Learning Objectives

By the end of this lesson, you will be able to:

1. Explain **why relational databases outperform spreadsheets** for structured data with relationships, and name the three update/insert/delete anomalies that normalization prevents
2. Write **SQL queries** using the six core clauses: `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY`, `LIMIT` ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â and explain why SQL execution order differs from write order
3. Distinguish **INNER JOIN** from **LEFT JOIN** and choose the right one for each use case; debug common join mistakes using row-count sanity checks
4. Contrast **OLTP** (row-based, SQLite) with **OLAP** (column-based, DuckDB) and explain why column storage is faster for aggregation queries
5. Query data using both **`sqlite3`** and **`duckdb`**, including DuckDB's `read_csv_auto()` direct-CSV approach and its `EXCLUDE` keyword
6. Use the **`wrangle_nepal_data()`** reusable module to load clean, model-ready data in a single call and explain what each step does


## Part 1: Database Fundamentals

### Why databases beat spreadsheets

> ÃƒÂ¢Ã‚ÂÃ¢â‚¬Å“ Could you manage the Nepal earthquake survey ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â 234,835 buildings, four interlocked data sources, across four districts ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â in a spreadsheet?

Technically yes. Practically, you would hit Excel's row limit (1,048,576 rows for very large files), face data-duplication nightmares, and risk inconsistency the moment anyone edits two rows that should always match. Databases prevent these problems through **normalization** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â splitting data into related tables so each fact is stored exactly once.

**The redundancy problem ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a flat-file example:**

| Customer    | Address             | Product | Price   |
|-------------|---------------------|---------|---------|
| John Smith  | 123 Main St, Boston | Laptop  | \$999  |
| John Smith  | 123 Main St, Boston | Mouse   | \$25   |
| Jane Doe    | 456 Oak Ave, NYC    | Laptop  | \$999  |

Problems: John's address is stored twice. If he moves, you must update two rows ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â or introduce inconsistency. Delete his only order and you lose his address.

**The normalized version ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â two tables, zero redundancy:**

Customers Table:

| CustomerID | Name       | Address             |
|------------|------------|---------------------|
| 1          | John Smith | 123 Main St, Boston |
| 2          | Jane Doe   | 456 Oak Ave, NYC    |

Orders Table:

| OrderID | CustomerID | Product | Price  |
|---------|------------|---------|--------|
| 101     | 1          | Laptop  | \$999 |
| 102     | 1          | Mouse   | \$25  |
| 103     | 2          | Laptop  | \$999 |

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **Benefits of normalization:**
> - **No redundancy**: John's address stored once, referenced everywhere
> - **Consistency**: Update one row, every query instantly reflects the change across the entire database
> - **Integrity**: Constraints prevent orphaned orders ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â you cannot delete a customer who still has orders
> - **Clarity**: "How many customers named John?" is unambiguous ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â one row per customer

### Data integrity constraints

Databases enforce rules that prevent bad data from entering in the first place:

| Constraint | What it enforces | Example in Nepal data |
|------------|-----------------|----------------------|
| `PRIMARY KEY` | Each row has a unique, non-null identifier | `building_id` is unique in `building_structure` |
| `FOREIGN KEY` | Referenced rows must exist in the parent table | `id_map.building_id` must exist in `building_structure.building_id` |
| `NOT NULL` | A column cannot be empty | `damage_grade` cannot be NULL ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â every building must have a damage assessment |
| `UNIQUE` | Values in a column (or combination) must be distinct | A household can appear in `id_map` only once per building |
| `CHECK` | Values must satisfy a condition | `district_id IN (1, 2, 3, 4)` ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â only valid district IDs allowed |

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **Why constraints matter:** they make your data trustworthy before you run a single line of Python. A model trained on data with constraint violations (e.g., orphaned foreign keys) will produce silently wrong results. The database enforces correctness so you do not have to check it in Python.


### Basic database terminology

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ **Quick reference** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the terms you will encounter in every SQL context:

| Term | Definition | Nepal data example |
|------|-----------|-------------------|
| **Schema** | Blueprint of the database ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â which tables, columns, data types, and relationships exist | Our four-table design (structure, damage, demographics, id_map) |
| **Table** | Rows = records; columns = fields ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â like an Excel sheet with enforced types | `building_structure` (234,835 rows ÃƒÆ’Ã¢â‚¬â€ 9 columns) |
| **Primary Key (PK)** | Unique identifier per row; no duplicates, no NULLs allowed | `building_id` in `building_structure` |
| **Foreign Key (FK)** | Column that references another table's PK; creates the link between tables | `building_id` in `id_map` references `building_structure.building_id` |
| **Query** | A declarative request for data ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â you say *what* you want, not *how* to get it | `SELECT * FROM building_structure LIMIT 5` |
| **Normalization** | Splitting data into related tables to eliminate redundancy and anomalies | Separating building features from damage grades into separate tables |
| **Index** | A data structure the database builds automatically on PK/FK columns to speed up lookups | SQLite auto-indexes `building_id` when you define it as PK |
| **Transaction** | A group of operations that succeeds or fails together (ACID guarantee) | Inserting a new building and its damage record atomically |

> ÃƒÂ¢Ã…Â¾Ã‚Â¡ÃƒÂ¯Ã‚Â¸Ã‚Â Now that we know what a database is, let's see how **relationships between tables** work ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the feature that makes relational databases powerful.


### The three database anomalies ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â why normalization is not optional

When data is not normalized (stored as one big flat table), three types of problems emerge. Collectively, these are called the **modification anomalies**. Understanding them is the best argument for relational database design.

**1. Update anomaly**

If John Smith's address appears in 50 rows (one per order), updating it requires finding and changing all 50. Miss one, and the database now contains contradictory data.

```
Before update: 50 rows with "123 Main St, Boston"
After update:  49 rows say "456 Oak Ave, NYC" + 1 stale row still says "123 Main St, Boston"
ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Query for John's address now returns TWO different answers
```

> In the Nepal context: imagine storing district_name in every building row. When the government renames "Gorkha" to "Gorkha Metropolitan City," you must update 70,836 rows ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a massive operation that can fail partway through.

**2. Insert anomaly**

You cannot add a fact to the database unless you already know all non-null columns. In a flat orders table with address as a required field, you cannot record a new customer until they place their first order.

> In the Nepal context: in a fully denormalized table, you could not add a household demographic record without simultaneously providing a building and damage grade ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â even if you have not assessed the building yet.

**3. Delete anomaly**

Deleting one piece of information accidentally destroys another. Delete the last order for a customer, and you lose the customer's address entirely.

> In the Nepal context: if building features and damage grades are in the same row, deleting a building removes its demographic and damage history simultaneously ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â even if you only intended to remove it from the active survey.

**The solution: normalization**

By storing each fact exactly once in the table that *owns* it, normalization eliminates all three anomalies:
- Update: change John's address in one row ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ all queries instantly reflect it
- Insert: add a customer row without orders
- Delete: remove an order without losing the customer

### CRUD ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the four database operations

All database interaction reduces to four operations:

| Operation | SQL command | Description |
|-----------|------------|-------------|
| **Create** | `INSERT INTO` | Add a new row |
| **Read** | `SELECT` | Query one or more rows |
| **Update** | `UPDATE ... SET` | Modify existing rows |
| **Delete** | `DELETE FROM` | Remove rows |

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ **For this course, we only use Read (`SELECT`)** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â we are analyzing the Nepal earthquake data, not modifying it. But understanding that databases support CRUD operations explains why they are used everywhere, from mobile apps to enterprise systems.


## Part 2: Relational Thinking ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Understanding Relationships

The power of databases comes from **relationships**: tables are connected through shared keys, and those connections are what make JOINs possible.

### The three types of relationships

| Type | Direction | Example | Description |
|------|-----------|---------|-------------|
| **One-to-Many** (most common) | 1 ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ many | One customer ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ many orders | Each order belongs to exactly one customer; one customer can place many orders |
| **Many-to-Many** | many ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬Â many | Students ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬Â courses | A student takes many courses; a course has many students (requires a bridge/junction table) |
| **One-to-One** (rare) | 1 ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ 1 | One person ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ one passport | Each entity on both sides appears at most once |

**Nepal earthquake data relationships:**

- **1 building ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ 1 damage assessment** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â one-to-one relationship via `building_id` (every building has exactly one damage grade)
- **1 building ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ many households** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â one-to-many relationship via `id_map` (apartment buildings can have multiple households)
- **Households ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬Â buildings** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â many-to-many relationship mediated by `id_map`, which acts as the bridge/junction table

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **Why relationships matter for our analysis:** to answer "What is the severe damage rate for Gurung-majority buildings in Gorkha?", you must traverse all four tables ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â building features, damage grades, household demographics, and the map that connects buildings to districts. Without the relationship structure, this query is impossible from any single table.

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **The bridge table pattern:** when two tables have a many-to-many relationship, databases use a third table ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a **bridge** or **junction** table ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â that holds both foreign keys. `id_map` is exactly this: it holds `building_id` (FK ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ building_structure) and `household_id` (FK ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ household_demographics), plus the `district_id` that tells us which administrative district the building is in.


### Entity-Relationship (ER) Diagrams

An **ER diagram** shows tables, their columns, and the connections between them visually. Learning to read one is a core skill you will use every time you approach an unfamiliar database.

**Key symbols:**

| Symbol | Meaning |
|--------|---------|
| ÃƒÂ¢Ã‚Â­Ã‚Â **PK** | Primary Key ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the unique row identifier |
| ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ¢â‚¬â€ **FK** | Foreign Key ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the column that points to another table's PK |
| **1** and **Many** labels | The cardinality of the relationship |
| Arrow direction | Shows which table "owns" each side of the relationship |
| **Solid line** | Identifying relationship (child cannot exist without parent) |
| **Dashed line** | Non-identifying relationship (child can exist independently) |

**A simple ER diagram ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Customers and Orders:**

```
ÃƒÂ¢Ã¢â‚¬ÂÃ…â€™ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡  Customers   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ…â€œÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¤
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ¢Ã‚Â­Ã‚Â ID (PK)   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ Name         ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ Address      ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬ÂÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‹Å“
       ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ 1     (One customer can haveÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¦)
       ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
       ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ Many
       ÃƒÂ¢Ã¢â‚¬â€œÃ‚Â¼
ÃƒÂ¢Ã¢â‚¬ÂÃ…â€™ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡   Orders     ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ…â€œÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¤
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ¢Ã‚Â­Ã‚Â ID (PK)   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ¢â‚¬â€ CustID(FK)ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ references Customers.ID
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ Product      ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ Price        ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬ÂÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‹Å“
```

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ **How to read any ER diagram:**
> 1. Find the primary keys (ÃƒÂ¢Ã‚Â­Ã‚Â) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â these are the unique identifiers for each table
> 2. Trace the foreign keys (ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ¢â‚¬â€) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â these arrows show the relationships
> 3. Read the cardinality labels (1 vs Many) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â they tell you how many rows on each side can participate
> 4. Follow the arrows ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a `1` on the parent side and `Many` on the child side is a one-to-many relationship

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **Why ER diagrams matter for writing JOINs:** every JOIN you write follows an arrow in the ER diagram. If you can read the ER diagram, you can reconstruct any JOIN without guessing.

> ÃƒÂ¢Ã…Â¾Ã‚Â¡ÃƒÂ¯Ã‚Â¸Ã‚Â Before we can use relationships in queries, we need to understand **the order in which SQL evaluates a query** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â which is not the same order in which you write it.


## Part 3: SQL Execution Order

You write SQL in this order:

```sql
SELECT name, age
FROM customers
WHERE age > 21
GROUP BY city
ORDER BY age DESC
LIMIT 10
```

But the database **executes** it in a different order:

| Step | Clause | What happens |
|------|--------|-------------|
| 1 | `FROM` | Open the table and load rows (or resolve the JOIN) |
| 2 | `WHERE` | Filter rows that satisfy the condition ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â happens on individual rows |
| 3 | `GROUP BY` | Group remaining rows by the specified column |
| 4 | `HAVING` | Filter groups (like WHERE, but applies after grouping) |
| 5 | `SELECT` | Choose which columns to show and evaluate expressions |
| 6 | `ORDER BY` | Sort the result set |
| 7 | `LIMIT` | Keep only the first N rows |

> ÃƒÂ¢Ã…Â¡Ã‚Â ÃƒÂ¯Ã‚Â¸Ã‚Â **Why execution order matters ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â three common pitfalls:**
>
> 1. **Aliases in WHERE:** you cannot use a `SELECT` alias in a `WHERE` clause ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the alias does not exist yet when `WHERE` runs. Use a subquery or repeat the expression.
>
> 2. **GROUP BY constraints:** because `GROUP BY` runs before `SELECT`, every column in `SELECT` must either be in `GROUP BY` or wrapped in an aggregation function (`COUNT()`, `SUM()`, `AVG()`, etc.). `SELECT name, SUM(price) FROM orders GROUP BY city` fails ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â `name` is not in `GROUP BY` and not aggregated.
>
> 3. **WHERE vs HAVING:** `WHERE` filters *rows before* grouping; `HAVING` filters *groups after* grouping. Use `WHERE` to exclude individual buildings; use `HAVING` to exclude groups with fewer than N buildings.

**WHERE vs HAVING ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â side by side:**

```sql
-- WHERE: filter rows before grouping (efficient ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â fewer rows enter GROUP BY)
SELECT foundation_type, COUNT(*) AS n
FROM building_structure
WHERE district_id = 4          -- filter rows first
GROUP BY foundation_type

-- HAVING: filter groups after grouping
SELECT foundation_type, COUNT(*) AS n
FROM building_structure
GROUP BY foundation_type
HAVING COUNT(*) > 1000         -- keep only common foundation types
```

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **The mental model:** think of SQL execution as a pipeline ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â you open the table, filter it, group it, filter groups, then decide what to show. The *output* step (`SELECT`) always comes after all the *filtering* steps.

> ÃƒÂ¢Ã…Â¾Ã‚Â¡ÃƒÂ¯Ã‚Â¸Ã‚Â With the execution order in mind, we can now tackle the most powerful SQL operation: **JOIN** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â combining two tables into one result set using a shared key.


### Five SQL patterns you will use in every data science workflow

These five patterns appear repeatedly throughout L2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“L5. Learning to recognize and write them fluently will save you time in every project.

**Pattern 1: Count by category (GROUP BY + COUNT)**

The most common exploratory SQL query. Answers: "How many buildings of each type are there?"

```sql
SELECT foundation_type, COUNT(*) AS n_buildings
FROM building_structure
GROUP BY foundation_type
ORDER BY n_buildings DESC
```

> Use this to understand the distribution of categorical features ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the backbone of EDA.

**Pattern 2: Conditional aggregation (CASE WHEN)**

Compute multiple statistics in a single pass. Answers: "How many severe vs. non-severe buildings per foundation type?"

```sql
SELECT foundation_type,
       COUNT(*) AS total,
       SUM(CASE WHEN d.damage_grade IN ('Grade 4', 'Grade 5') THEN 1 ELSE 0 END) AS severe,
       ROUND(100.0 * SUM(CASE WHEN d.damage_grade IN ('Grade 4', 'Grade 5') THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_severe
FROM building_structure s
JOIN building_damage d ON s.building_id = d.building_id
GROUP BY foundation_type
ORDER BY pct_severe DESC
```

> This is how you compute group-level rates without loading data into pandas first.

**Pattern 3: Subquery for filtered aggregation**

Run a query whose result depends on the output of another query. Answers: "Which buildings have above-average age?"

```sql
SELECT building_id, age_building
FROM building_structure
WHERE age_building > (SELECT AVG(age_building) FROM building_structure)
ORDER BY age_building DESC
```

> Subqueries appear inside `WHERE`, `FROM`, or `SELECT`. The inner query runs first; the outer query uses its result.

**Pattern 4: Multi-table JOIN chain**

Combine all four Nepal tables in one query ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the foundation of L4.

```sql
SELECT s.building_id, s.foundation_type, d.damage_grade,
       i.district_id, h.caste_household
FROM building_structure s
INNER JOIN building_damage d   ON s.building_id = d.building_id
INNER JOIN id_map i            ON s.building_id = i.building_id
INNER JOIN household_demographics h ON i.household_id = h.household_id
WHERE i.district_id = 4
LIMIT 10
```

> Each JOIN follows one arrow in the ER diagram. The alias pattern (`s`, `d`, `i`, `h`) prevents ambiguity when multiple tables have columns with the same name.

**Pattern 5: HAVING for group filtering**

Filter after aggregation. Answers: "Which districts have severe damage rate above 50%?"

```sql
SELECT i.district_id,
       COUNT(*) AS total_buildings,
       ROUND(100.0 * SUM(CASE WHEN d.damage_grade IN ('Grade 4', 'Grade 5') THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_severe
FROM building_structure s
JOIN building_damage d ON s.building_id = d.building_id
JOIN id_map i ON s.building_id = i.building_id
GROUP BY i.district_id
HAVING pct_severe > 50
```

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ **A note on SQL dialect:** some of these patterns use DuckDB-specific features (like referencing a `SELECT` alias in `HAVING`). In standard SQL, you must repeat the full expression in `HAVING`. DuckDB is more permissive and data-science-friendly.


## Part 4: Understanding Joins ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â The Foundation of Relational Databases

A **join** combines rows from two tables by **matching a shared key column**. This is what makes it possible to answer questions like "What damage grade does building 12345 have?" when the answer lives in a different table from the building's physical features.

### The matching key concept

When we say "join on a key," we mean:
- Table A has a column called `building_id`
- Table B has a column called `building_id`
- We tell the database: "for each row in A, find rows in B where these two values are equal"

```sql
ON building_structure.building_id = building_damage.building_id
        ÃƒÂ¢Ã¢â‚¬â€œÃ‚Â²                                    ÃƒÂ¢Ã¢â‚¬â€œÃ‚Â²
   Left table column              Right table column
   (must hold the same values!)
```

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ The column **names** do not have to match ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â you could have `id` on one side and `building_id` on the other. But the **values** must match. Always join on the Foreign Key ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Primary Key relationship; joining on arbitrary columns produces meaningless results (a Cartesian product).

### Step-by-step visual example

**Customers (left table):**

| ID | Name | City |
|----|------|------|
| 1  | John | NYC  |
| 2  | Jane | LA   |
| 3  | Bob  | NYC  |

**Orders (right table):**

| OrderID | CustomerID | Product  |
|---------|------------|----------|
| 101     | 1          | Laptop   |
| 102     | 1          | Mouse    |
| 103     | 2          | Keyboard |
| 104     | 4          | Monitor  |

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ CustomerID = 4 in Orders does not exist in Customers. This edge case is what distinguishes INNER JOIN from LEFT JOIN ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the two join types give different answers for Customer ID 3 (Bob) and Order ID 104 (Monitor).


### INNER JOIN: the intersection

**What it does:** returns **only rows where the matching key exists in BOTH tables**.

```sql
SELECT customers.name, orders.product
FROM customers
INNER JOIN orders ON customers.id = orders.customerid
```

**Step-by-step matching:**

```
Customer ID 1 (John): found orders 101, 102  ÃƒÂ¢Ã…â€œÃ¢â‚¬Å“ INCLUDE (2 rows in result)
Customer ID 2 (Jane): found order 103         ÃƒÂ¢Ã…â€œÃ¢â‚¬Å“ INCLUDE (1 row in result)
Customer ID 3 (Bob):  found no orders         ÃƒÂ¢Ã…â€œÃ¢â‚¬â€ EXCLUDE (silently dropped)
Order ID 104 (Monitor): customer 4 not found  ÃƒÂ¢Ã…â€œÃ¢â‚¬â€ EXCLUDE (silently dropped)
```

**Result:**

| Name | Product  |
|------|----------|
| John | Laptop   |
| John | Mouse    |
| Jane | Keyboard |

Bob and the Monitor order are silently dropped from the result.

> ÃƒÂ°Ã…Â¸Ã…Â¡Ã‚Â¦ **When to use INNER JOIN:**
> - You only want records that exist on **both sides** of the join
> - You want to eliminate rows with missing relationships
> - Nepal example: "Find all buildings that **have** a damage assessment"

```sql
SELECT s.building_id, s.age_building, d.damage_grade
FROM building_structure s
INNER JOIN building_damage d ON s.building_id = d.building_id
-- Returns only buildings appearing in BOTH tables (all 234,835 matched records)
```

> ÃƒÂ¢Ã…Â¡Ã‚Â ÃƒÂ¯Ã‚Â¸Ã‚Â **INNER JOIN silently loses data.** If your query returns fewer rows than expected, you may be inadvertently dropping buildings whose key does not match. Always cross-check row counts after a JOIN ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â if `building_structure` has 234,835 rows and your INNER JOIN result has fewer, investigate why.

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **Row count check:** `SELECT COUNT(*) FROM building_structure` should equal `SELECT COUNT(*) FROM building_structure INNER JOIN building_damage ON ...`. If it doesn't, you have unmatched rows worth investigating.


### LEFT JOIN: keep everything from the left

**What it does:** returns **ALL rows from the left table**, plus matching rows from the right ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â showing `NULL` where no match exists.

```sql
SELECT customers.name, orders.product
FROM customers
LEFT JOIN orders ON customers.id = orders.customerid
```

**Result:**

| Name | Product  |
|------|----------|
| John | Laptop   |
| John | Mouse    |
| Jane | Keyboard |
| Bob  | NULL     |

Bob appears with `NULL` in the Product column ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â he has no orders, but he is still in the result. The Monitor order (customer ID 4 does not exist in the left table) is still excluded.

> ÃƒÂ°Ã…Â¸Ã…Â¡Ã‚Â¦ **When to use LEFT JOIN:**
> - You want **all records from the left table**, whether or not they have a match
> - You want to **identify missing relationships** (`NULL` values reveal data quality problems)
> - Nepal example: "Which buildings have **no** damage assessment?" (data quality check)

```sql
SELECT s.building_id, d.damage_grade
FROM building_structure s
LEFT JOIN building_damage d ON s.building_id = d.building_id
WHERE d.damage_grade IS NULL  -- NULL means: this building has no matching damage record
```

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **NULL as a diagnostic tool:** if the LEFT JOIN + `IS NULL` query returns any rows, those are buildings with missing damage assessments ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a data quality issue. For our Nepal dataset, every building should have exactly one damage record, so this query should return 0 rows. If it returns rows, something went wrong in the data pipeline.

### INNER JOIN vs LEFT JOIN ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â comparison table

| Scenario | Use | Why |
|----------|-----|-----|
| Customers **with** orders only | INNER JOIN | Want rows that match on both sides |
| Customers **with or without** orders | LEFT JOIN | Keep all left rows; see who has no orders |
| All buildings **with** damage data | INNER JOIN | Only complete records |
| All buildings, show missing damage | LEFT JOIN | Identify data quality issues |
| Multi-table chain | Depends per join | Each JOIN in a chain can independently be INNER or LEFT |

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **For the Nepal project:** we use INNER JOIN in most analyses because the data has been pre-filtered ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â every building has a matched damage assessment. LEFT JOIN becomes useful in L4 when verifying join completeness across all four tables.

> ÃƒÂ¢Ã…Â¾Ã‚Â¡ÃƒÂ¯Ã‚Â¸Ã‚Â Now that we understand *how* joins work, let's look at *why* some databases are dramatically faster at joins and aggregations ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the OLTP vs OLAP distinction.


### JOIN debugging ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â what goes wrong and how to fix it

Joins are the most common source of subtle data errors in data science. Here is a systematic checklist for diagnosing and fixing join problems.

**Problem 1: Row count explosion (Cartesian product)**

*Symptom:* your result has far more rows than expected ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â often `len(df) = left_rows ÃƒÆ’Ã¢â‚¬â€ right_rows`.

*Cause:* joining on a key that has duplicate values on both sides, creating every combination.

```sql
-- Bad: id_map has multiple rows per building_id (one per household)
SELECT s.*, i.district_id
FROM building_structure s
JOIN id_map i ON s.building_id = i.building_id
-- Result: 249,932 rows, not 234,835! Each building matches multiple households.

-- Fix: use DISTINCT or pre-aggregate before joining
SELECT DISTINCT s.building_id, i.district_id
FROM building_structure s
JOIN id_map i ON s.building_id = i.building_id
-- OR: join on a pre-deduplicated version of id_map
```

**Problem 2: Silently missing rows**

*Symptom:* your result has fewer rows than expected.

*Cause:* INNER JOIN drops unmatched rows silently.

```sql
-- Diagnostic: count rows before and after join
SELECT COUNT(*) FROM building_structure;  -- 234,835
SELECT COUNT(*) FROM building_structure s
INNER JOIN building_damage d ON s.building_id = d.building_id;  -- should still be 234,835

-- If count drops, find the unmatched rows with LEFT JOIN + IS NULL
SELECT s.building_id
FROM building_structure s
LEFT JOIN building_damage d ON s.building_id = d.building_id
WHERE d.building_id IS NULL;
```

**Problem 3: Ambiguous column names**

*Symptom:* `AmbiguousColumnException: building_id is ambiguous` or incorrect values in result.

*Cause:* both tables have a column with the same name and you didn't qualify which one you want.

```sql
-- Bad: building_id appears in both tables
SELECT building_id, age_building, damage_grade
FROM building_structure
JOIN building_damage ON building_structure.building_id = building_damage.building_id;

-- Good: use table aliases to qualify every ambiguous column
SELECT s.building_id, s.age_building, d.damage_grade
FROM building_structure s
JOIN building_damage d ON s.building_id = d.building_id;
```

**Problem 4: Wrong join key**

*Symptom:* result looks correct but produces wrong values (silent error).

*Cause:* joining on a non-primary-key column that happens to have matching values.

> **Rule:** always join on PK ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ FK relationships. Trace the arrow in the ER diagram. If the ER diagram shows `building_structure.building_id ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ building_damage.building_id`, that is your join key.

**Quick JOIN sanity-check protocol:**

1. Count rows in each source table before joining
2. Count rows in join result ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â do they match expectations?
3. Sample 5-10 rows and manually verify the join is correct
4. For multi-table chains: verify one join at a time, building up to the full query


## Part 5: OLTP vs OLAP ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Two Different Database Philosophies

Not all databases are optimized for the same job. The two dominant paradigms serve fundamentally different workloads.

| | **OLTP** (Online Transaction Processing) | **OLAP** (Online Analytical Processing) |
|---|---|---|
| **Optimized for** | Fast writes + small, targeted reads | Fast reads over millions of rows |
| **Typical query** | "Get account balance for user 12345" | "Average balance by age group across 1M users" |
| **Consistency** | ACID (Atomic, Consistent, Isolated, Durable) | Read-optimized; no write consistency needed |
| **Storage** | **Row-based**: full row in memory per transaction | **Column-based**: only the needed columns in memory |
| **Example tools** | SQLite, MySQL, PostgreSQL | DuckDB, BigQuery, Snowflake, ClickHouse |
| **Nepal project role** | `nepal.sqlite` ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â queried in L1 for comparison | DuckDB on CSV ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â powers all modeling in L2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“L5 |

**The storage format analogy:**

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **Row storage** is like a filing cabinet where each folder holds one person's complete record ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â fast to retrieve one person's full profile, slow to scan everyone's age across 1M records.
>
> **Column storage** is like a spreadsheet where one column holds everyone's age ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â fast to compute aggregate statistics, slow to assemble one complete record from scratch.

**Real-world examples:**

| Use case | Tool | Why |
|----------|------|-----|
| Mobile banking app (user checks balance) | OLTP / SQLite | Fast write/update per user; ACID transactions required |
| Analytics dashboard (average balance by age group) | OLAP / DuckDB | Scans millions of rows; only needs 2 columns |
| Website session tracking | OLTP / MySQL | High write volume, small individual reads |
| Data science EDA (GROUP BY, histograms) | OLAP / DuckDB | Aggregate scans over large datasets |
| Mobile app local database | OLTP / SQLite | Zero-configuration embedded database |
| Machine learning feature engineering | OLAP / DuckDB | Multi-column aggregations over large CSVs |

**When to choose which:**

- **Choose OLTP (SQLite/PostgreSQL/MySQL) when:** your application makes frequent small writes, needs transaction guarantees (bank transfers must be atomic), or runs embedded in a device or app
- **Choose OLAP (DuckDB/BigQuery/Snowflake) when:** you are analyzing historical data in batch, running GROUP BY / aggregation queries, or processing large CSV/Parquet files for machine learning

> ÃƒÂ¢Ã…Â¾Ã‚Â¡ÃƒÂ¯Ã‚Â¸Ã‚Â Let's go one level deeper: *why* does column storage give DuckDB such a large speed advantage for the analytical queries data scientists run?


## Part 6: Row Storage vs Column Storage ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Why DuckDB Wins at Analytics

### The core trade-off

| Storage | Memory layout | Fast for | Slow for |
|---------|---------------|----------|----------|
| **Row (SQLite)** | `[ID|Name|Age|City]` per record | Reading one complete record | Summing one column over all rows |
| **Column (DuckDB)** | `[All IDs]` then `[All Names]` then `[All Ages]` | Aggregating one column over all rows | Assembling one complete record |

### Three mechanisms behind DuckDB's speed

**1. Memory bandwidth savings**

Modern CPUs can only process data that is in memory. Loading data from disk into memory is the primary bottleneck for analytical queries.

```
Query: SELECT SUM(age) FROM customers WHERE city = 'NYC'

SQLite:  Load 1M rows ÃƒÆ’Ã¢â‚¬â€ 10 columns = 10M values into memory (you only use 2 columns)
DuckDB:  Load Age column (1M values) + City column (1M values) = 2M values total
         ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ 80% less memory bandwidth wasted
```

> The key insight: if your query only touches 2 out of 10 columns, row storage wastes 80% of every memory fetch. Column storage fetches only what you need.

**2. Vectorization (SIMD ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Same Instruction, Multiple Data)**

```
Traditional (SQLite):  if (city[0]=='NYC') sum += age[0];  ÃƒÂ¢Ã¢â‚¬Â Ã‚Â 1 row per instruction
Vectorized (DuckDB):   check 128 cities at once, add 128 ages at once  ÃƒÂ¢Ã¢â‚¬Â Ã‚Â 128ÃƒÆ’Ã¢â‚¬â€ throughput
```

Modern CPUs have special registers that hold 128+ numbers simultaneously. DuckDB leverages SIMD instructions to process entire *batches* in one CPU instruction. SQLite processes one row at a time; DuckDB processes 128+ rows simultaneously.

**3. Query optimization**

DuckDB's query optimizer reorders your query to filter as early as possible, reducing work before the expensive aggregation:

```
Your SQL:         SELECT SUM(age) FROM customers WHERE city = 'NYC'
DuckDB executes:  FILTER city='NYC'  ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢  SUM remaining ages  (reduces 1M to ~100K first)
SQLite path:      Scan all 1M rows  ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢  SUM  ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢  filter        (less optimal order)
```

**4. Compression benefits**

Column storage enables much better compression than row storage, because a column of similar values compresses more efficiently than a mixed-type row:

- An `age` column full of integers: compresses to ~10% of original size
- A row with `[string, int, float, bool, string]`: minimal compression benefit

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **The result:** for GROUP BY queries over 234,835 buildings, DuckDB can be 10ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“100ÃƒÆ’Ã¢â‚¬â€ faster than SQLite. On our dataset the wall-clock difference is modest (sub-second on fast hardware), but on multi-million-row production datasets the gap is dramatic. This is why Lessons 2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5 use DuckDB exclusively.

> ÃƒÂ¢Ã…Â¾Ã‚Â¡ÃƒÂ¯Ã‚Â¸Ã‚Â Now let's look at the specific Nepal earthquake tables we will be querying, and how their relationships are structured in the ER diagram.


## Part 7: The Nepal Earthquake Data ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Our Relational Context

### The four tables

| Table | Rows | Description | Key columns |
|-------|------|-------------|-------------|
| **`building_structure`** | 234,835 | Physical features of each building | `building_id` (PK), `age_building`, `foundation_type`, `roof_type`, `plinth_area_sq_ft`, `height_ft_pre_eq`, `ground_floor_type`, `other_floor_type`, `position` |
| **`building_damage`** | 234,835 | Post-earthquake damage assessment | `building_id` (PK/FK), `damage_grade` (Grades 1ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5), `technical_solution_proposed` |
| **`household_demographics`** | 249,932 | Information about building occupants | `household_id` (PK), `caste_household`, `gender_household_head`, `count_floors_pre_eq` |
| **`id_map`** | 249,932 | Bridge table linking buildings to households | `household_id` (FK), `building_id` (FK), `district_id`, `vdcmun_id` |

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ **The row counts are telling:**
> - 234,835 buildings each have exactly **one** damage record (one-to-one relationship via `building_id`)
> - 249,932 households occupy those 234,835 buildings ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â meaning some buildings house multiple households (one building ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ many households relationship via `id_map`)
> - `id_map` has 249,932 rows because it has one row per householdÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“building pair, not one row per building

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **Why four tables instead of one?** Each table has a single, clear responsibility:
> - `building_structure`: physical characteristics of the building
> - `building_damage`: outcome of the earthquake assessment
> - `household_demographics`: who lives in the building
> - `id_map`: the administrative geography (which district) and the household-to-building bridge
>
> Keeping them separate prevents the three anomalies discussed in Part 1 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â update, insert, and delete anomalies. It also means that a query about building features does not need to load demographic data (and vice versa), which improves query performance.


### ER diagram for Nepal data

```
ÃƒÂ¢Ã¢â‚¬ÂÃ…â€™ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    BUILDING_STRUCTURE        ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ…â€œÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¤
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ¢Ã‚Â­Ã‚Â building_id (PK)          ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    age_building              ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    foundation_type           ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    roof_type / plinth_area   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬ÂÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‹Å“
       ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ 1 (one building has...)
       ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
   ÃƒÂ¢Ã¢â‚¬ÂÃ…â€™ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â´ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â
   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡                               ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ 1 (one damage record)         ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ Many (many households via id_map)
   ÃƒÂ¢Ã¢â‚¬â€œÃ‚Â¼                               ÃƒÂ¢Ã¢â‚¬â€œÃ‚Â¼
ÃƒÂ¢Ã¢â‚¬ÂÃ…â€™ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â   ÃƒÂ¢Ã¢â‚¬ÂÃ…â€™ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡   BUILDING_DAMAGE    ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡      ID_MAP           ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ…â€œÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¤   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡   (bridge table)      ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ¢Ã‚Â­Ã‚Â/ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ¢â‚¬â€ building_id    ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡   ÃƒÂ¢Ã¢â‚¬ÂÃ…â€œÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¤
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡         (PK/FK)      ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ¢â‚¬â€ building_id (FK)  ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    damage_grade      ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ¢â‚¬â€ household_id (FK) ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    tech_solution     ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    district_id        ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬ÂÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‹Å“   ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬ÂÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‹Å“
                                      ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ Many
                                      ÃƒÂ¢Ã¢â‚¬â€œÃ‚Â¼
                           ÃƒÂ¢Ã¢â‚¬ÂÃ…â€™ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â
                           ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ HOUSEHOLD_DEMOGRAPHICSÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
                           ÃƒÂ¢Ã¢â‚¬ÂÃ…â€œÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‚Â¤
                           ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡ ÃƒÂ¢Ã‚Â­Ã‚Â household_id (PK)  ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
                           ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    caste_household    ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
                           ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡    gender_hh_head     ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬Å¡
                           ÃƒÂ¢Ã¢â‚¬ÂÃ¢â‚¬ÂÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ‹Å“
```

**How to answer "Where is damage worst?" requires joining all four tables:**

```sql
SELECT s.building_id, d.damage_grade, i.district_id, h.caste_household
FROM building_structure s
JOIN building_damage d ON s.building_id = d.building_id     -- structure ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ damage
JOIN id_map i ON s.building_id = i.building_id              -- structure ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ district
JOIN household_demographics h ON i.household_id = h.household_id  -- map ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ demographics
WHERE i.district_id = 4  -- Gorkha district
```

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ **Follow the arrows:** every JOIN in this query follows one arrow in the ER diagram above. The ER diagram is your JOIN recipe ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â if you can read it, you can write any multi-table query.


### Understanding the four districts

Our survey covers four districts in Nepal. For most of Lessons 2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5, we focus on **Gorkha** (district_id = 4), the earthquake's epicenter.

| district_id | District | Buildings | Notes |
|-------------|----------|-----------|-------|
| 1 | Sindhupalchok | ~36,000 | North of Kathmandu; heavily affected |
| 2 | Okhaldhunga | ~55,000 | Eastern hilly district |
| 3 | Kavrepalanchok | ~73,000 | Adjacent to Kathmandu valley |
| 4 | **Gorkha** | **~70,836** | **Epicenter district; Gurung-dominant** |

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **Domain trick for data validation:** if you query caste distribution by district and see Gurung as the top group, you are looking at Gorkha. This demographic signature lets you verify that a district filter is working correctly without memorizing the `district_id` number. Lessons 2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5 filter to `district_id = 4` ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â always run this validation check after loading data.

**Code 4.1.2.1**: Query caste distribution by district to identify the districts and validate the Gorkha signature


In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('./nepal.sqlite')

# Query caste distribution by district
query = """
    SELECT i.district_id, h.caste_household, COUNT(*) as count
    FROM household_demographics h
    JOIN id_map i ON h.household_id = i.household_id
    GROUP BY i.district_id, h.caste_household
    ORDER BY i.district_id, count DESC
"""

df_caste = pd.read_sql_query(query, conn)
conn.close()

# Show top 3 castes per district
for district in [1, 2, 3, 4]:
    print(f"\nDistrict {district}:")
    top_castes = df_caste[df_caste['district_id'] == district].head(3)
    for _, row in top_castes.iterrows():
        print(f"  {row['caste_household']}: {row['count']:,} households")

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  **Reading the output:** district 4 (Gorkha) is the only district where **Gurung** is the top caste. Other districts are Chhetree- or Tamang-dominant. This demographic fingerprint is useful for validating district-level queries: if you ever see an unexpected caste profile in your results, it signals that a district filter may have slipped or been applied incorrectly.

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **Key insight:** using domain knowledge to validate query results is good data science practice. You should always ask "does this output make sense given what I know about the data?" before trusting downstream analysis. In L4, we return to caste analysis in depth ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the equity question of *who* was hit hardest is as important as the predictive question of *what* predicts damage.

---

## Part 8: Setting Up the Python Environment

Let us import the libraries we will use throughout this lesson. Each library has a specific role in our database workflow:

- **`sqlite3`**: Python's built-in SQLite connector ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â no installation needed
- **`pandas`**: for converting SQL results into DataFrames
- **`duckdb`**: the fast analytical engine for CSV queries
- **`time`**: for measuring and comparing query execution times

**Code 4.1.0.1**: Import required libraries and verify versions


In [ ]:
import pandas as pd
import sqlite3
import duckdb
import numpy as np
import time

# Set display options
# pd.set_option('display.max_columns', None)

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169845682", h="3298dbabb7", width=700, height=450)

---

## Part 9: Working with SQLite

SQLite is the standard embedded relational database for Python. It requires no server, no installation beyond the Python standard library ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the entire database lives in a single file (`nepal.sqlite`). SQLite is the right tool when you need ACID transactions, embedded deployment, or compatibility with existing relational tooling.

### Six common SQL clauses ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â quick reference

| Clause | Purpose | Example |
|--------|---------|---------|
| **`SELECT`** | Choose columns to return | `SELECT building_id, age_building` |
| **`FROM`** | Specify the table | `FROM building_structure` |
| **`WHERE`** | Filter rows (pre-aggregation) | `WHERE district_id = 4` |
| **`GROUP BY`** | Aggregate rows by a column | `GROUP BY damage_grade` |
| **`ORDER BY`** | Sort the result | `ORDER BY age_building DESC` |
| **`LIMIT`** | Cap the number of rows returned | `LIMIT 5` |

### Connecting to SQLite ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the three-step pattern

Working with SQLite in Python always follows three steps:

```python
# Step 1: Connect ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â open (or create) the database file
conn = sqlite3.connect('./nepal.sqlite')

# Step 2: Query ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â execute SQL and get a DataFrame
df = pd.read_sql_query("SELECT * FROM building_structure LIMIT 5", conn)

# Step 3: Close ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â release the file handle
conn.close()
```

> ÃƒÂ¢Ã…Â¡Ã‚Â ÃƒÂ¯Ã‚Â¸Ã‚Â **Always call `conn.close()`** after you are done querying. Leaving connections open can lock the database file and cause errors in subsequent cells. In production code, use a `with` statement (`with sqlite3.connect(...) as conn:`) which automatically closes the connection even if an error occurs.

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **`pd.read_sql_query()` vs `cursor.execute()`:** `pd.read_sql_query()` is the data science-friendly wrapper ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â it executes the SQL and returns a DataFrame directly. The lower-level `cursor.execute()` returns raw tuples. We use `pd.read_sql_query()` throughout this course.

### ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã‚Â¹ Querying SQL Video


In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169845503", h="3298dbabb7", width=700, height=450)

> ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ‚Â **What to look for:** `sqlite_master` is SQLite's internal metadata table. Querying it with `type='table'` returns the names of all user-created tables in the database. You should see exactly four tables: `building_structure`, `building_damage`, `household_demographics`, and `id_map`. If any are missing, the `.sqlite` file may be incomplete or from the wrong version of the dataset.

**Code 4.1.3.1**: Connect to `./nepal.sqlite` and list all tables in the database


In [ ]:
# 1. Connect to the SQLite database
conn = sqlite3.connect('./nepal.sqlite')

# 2. Query to get all table names
# sqlite_master is a metadata table that contains information about the database schema
tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
table_names_list = tables_df['name'].tolist()

# 3. Close the connection
conn.close()

print(f"Tables in database: {table_names_list}")

### Basic queries ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â exploring the `building_structure` table

Now that we have confirmed the four tables exist, let us run basic queries to understand the data. The next tasks follow a natural exploration flow:

1. **Head query** (`LIMIT 5`) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â understand what columns exist and what values look like
2. **Column selection** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â practice `SELECT` with specific column names
3. **Aggregate query** (`COUNT(*)`) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â understand the size of the dataset
4. **DISTINCT query** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â understand the unique values in a categorical column
5. **Filtered query** (`WHERE`) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â practice selecting rows for a specific district

> ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ‚Â **What to look for in `building_structure`:** the column names tell you what physical features were recorded for each building ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â age, floor count, plinth area, foundation type, roof type, ground floor type. These are the features (X variables) we will use as predictors in Lessons 2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5. The Y variable (`severe_damage`) comes from `building_damage.damage_grade`.

**Code Task 4.1.3.2**: Query the `building_structure` table to get the first 5 rows using SQLite. Store the result in `df_sqlite_head`.


In [ ]:
conn = sqlite3.connect('./nepal.sqlite')

df_sqlite_head = pd.read_sql_query(
    "SELECT * FROM id_map LIMIT 5;",
    conn,
)

conn.close()
print(df_sqlite_head)

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  **Reading the `building_structure` output:** each row is one building. Key columns to notice:
> - `building_id`: integer primary key ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the join key for all multi-table queries
> - `age_building`: how old the building was at the time of the earthquake ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â older structures may be more vulnerable
> - `foundation_type`: the construction material of the foundation ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â one of the most predictive features for damage (stone, mud, RC, etc.)
> - `condition_post_eq`: verbal description of the building's state after the earthquake ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â **important note:** this column describes damage but is NOT our target; our target comes from `building_damage.damage_grade`
> - `superstructure`: the primary wall material (stone, RC, wood, etc.) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â critical for damage prediction
> - `plinth_area_sq_ft`: the footprint size of the building ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â larger buildings may distribute load differently


**Code Task 4.1.3.3**: Select only the `building_id` and `age_building`
columns from the `building_structure` table, limiting to the first 5
rows. Store in `df_sqlite_columns`.

In [ ]:
conn = sqlite3.connect('./nepal.sqlite')

df_sqlite_columns = pd.read_sql_query("""
  SELECT building_id, age_building
  FROM building_structure
  LIMIT 5
""", conn)


conn.close()
print(df_sqlite_columns)

**Code Task 4.1.3.4**: Count how many buildings are in the database
using SQLite. Store the result in `total_buildings_sqlite`.

- `COUNT(*)`: An aggregation function that counts rows

In [ ]:
conn = sqlite3.connect('./nepal.sqlite')

count_df = pd.read_sql_query(
    "SELECT COUNT(*) as count FROM building_structure",
    conn
)
total_buildings_sqlite = count_df.loc[0, 'count']

conn.close()
print(f"Total buildings: {total_buildings_sqlite}")

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  **234,835 buildings** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the full multi-district survey. This number matters for data validation: when you later filter to `district_id = 4`, you should get approximately 70,836 buildings (Gorkha). The full dataset spans four districts; 234,835 is the total across all four. If your filtered count is significantly different from 70,836, your district filter may be incorrect.


**Code Task 4.1.3.5**: Find all unique foundation types using SQLite.
Store in `df_foundation_types_sqlite`.

- `DISTINCT(...)`: Returns only unique values, removing duplicates

In [ ]:
conn = sqlite3.connect('./nepal.sqlite')

df_foundation_types_sqlite = pd.read_sql_query("""
    SELECT DISTINCT foundation_type
    FROM building_structure
""", conn)

conn.close()
print(df_foundation_types_sqlite)

**Code Task 4.1.3.6**: Select all columns from the `id_map` table,
showing only rows where the `district_id` is 4 (Gorkha) and limiting to
the first 5 rows. Store in `df_gorkha_sqlite`.

In [ ]:
conn = sqlite3.connect('./nepal.sqlite')

df_gorkha_sqlite = pd.read_sql_query("""
    SELECT *
    FROM id_map
    WHERE district_id = 4
    LIMIT 5
""", conn)

conn.close()
print(df_gorkha_sqlite)

ÃƒÂ¢Ã…â€œÃ¢â‚¬Â¦ **You may now attempt Multiple Choice Question 4.1.3.1** & **Multiple Choice Question 4.1.3.2**

---

## Part 10: Working with DuckDB

DuckDB takes a fundamentally different approach to querying data: **no connection object, no file to open, no close() call**. You simply write a SQL query and point it at a CSV file. DuckDB reads, processes, and returns results ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the data never needs to be loaded into a database first.

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **DuckDB's key superpower for data science:** `read_csv_auto()` lets you run analytical SQL queries directly on CSV files. This means you can answer questions like "How many buildings are in Gorkha?" without first importing the CSV into any database system. The CSV *is* the database.

**Three key differences from SQLite:**

| | SQLite | DuckDB |
|--|--------|--------|
| **Connection** | Explicit (`sqlite3.connect()`) + explicit close | No connection object for quick queries |
| **Data source** | `.sqlite` database file | CSV, Parquet, HDF5, or in-memory DataFrame |
| **Storage model** | Row-based (OLTP-optimized) | Column-based (OLAP-optimized; 10-100ÃƒÆ’Ã¢â‚¬â€ faster for analytics) |
| **Syntax** | Standard SQL | Standard SQL + `read_csv_auto()` + `EXCLUDE` + more |
| **Use case** | Embedded apps, ACID transactions | Analytics, data science, large CSV/Parquet files |
| **Setup** | Bundled with Python (`import sqlite3`) | `pip install duckdb` (not in standard library) |

**The DuckDB pattern:**

```python
import duckdb

# Query a CSV file directly ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â no import step needed
df = duckdb.sql('''
    SELECT * FROM read_csv_auto('./data/building_structure.csv') LIMIT 5
''').fetchdf()
```

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ `read_csv_auto()` infers column types automatically from the CSV headers. `.fetchdf()` converts the DuckDB result into a pandas DataFrame. Alternatively, `.df()` is an alias for `.fetchdf()`.

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **Why no connection object?** DuckDB's design philosophy is that for analytical queries, the overhead of managing a connection is unnecessary. You point it at a file, ask a question, get an answer. For repeated queries, DuckDB uses an in-memory database that persists across calls in the same Python session.

**Beyond CSV:** DuckDB can also query **Parquet** files (compressed columnar format, 5ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“10ÃƒÆ’Ã¢â‚¬â€ smaller than CSV and faster to read), **HDF5** files (hierarchical data storage used in scientific computing), and in-memory pandas DataFrames. This flexibility makes it a versatile workhorse for data science pipelines ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â you can query a pandas DataFrame with SQL, which is impossible in SQLite.

```python
# DuckDB can also query a pandas DataFrame directly!
import pandas as pd
df_existing = pd.read_csv('./data/building_structure.csv')
result = duckdb.sql("SELECT COUNT(*) FROM df_existing WHERE age_building > 50").fetchdf()
```

**Code 4.1.4.1**: Get the first 5 rows from the CSV file using DuckDB's `read_csv_auto()` capability


In [ ]:
# DuckDB can read CSV files directly!
df_duckdb_head = duckdb.sql("""
    SELECT * FROM read_csv_auto('./data/building_structure.csv') LIMIT 5
""").fetchdf()

print(df_duckdb_head)

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  **The output is identical to the SQLite result** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â same columns, same values, same 5 rows. This confirms that DuckDB's `read_csv_auto()` is reading the same underlying data as SQLite's `building_structure` table. The difference is not in the data, but in the *path to get it*: DuckDB required zero setup, zero connection management, and will be measurably faster on the aggregation queries we run later in this lesson. The `LIMIT 5` query is fast regardless of storage format ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the real speed difference appears with GROUP BY and aggregation queries over all 234,835 rows.


**Code Task 4.1.4.2**: Count how many buildings are in Gorkha (district
4) using DuckDB on the id_map CSV file. Store in `gorkha_count_duckdb`.

In [ ]:
count_result = duckdb.sql("""
    SELECT COUNT(*) as count
    FROM read_csv_auto('./data/id_map.csv')
    WHERE district_id = 4
""").fetchdf()

gorkha_count_duckdb = count_result.loc[0, 'count']
print(f"Buildings in Gorkha (District 4): {gorkha_count_duckdb}")

### DuckDB's `EXCLUDE` keyword ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the data science convenience

When loading data into pandas, it is common to want **all columns except one** (the ID column, which becomes the DataFrame index). Standard SQL requires you to list every column explicitly. DuckDB's `EXCLUDE` keyword makes this clean and readable:

```sql
SELECT DISTINCT building_id,    -- the column that becomes the index
       * EXCLUDE (building_id)  -- everything else, without repeating building_id
FROM read_csv_auto('./data/id_map.csv')
WHERE district_id = 4
```

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **Why `DISTINCT` on `building_id`?** The `id_map` table links buildings to households ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â one building can have many households. Using `DISTINCT` on `building_id` de-duplicates so each building appears once in the result. This is important for joining building-level features later: if we did not deduplicate, joining `id_map` to `building_structure` would multiply rows (Cartesian product problem from Fragment 9b).

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…â€™ **`EXCLUDE` is DuckDB-specific** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â standard SQL does not have this keyword. In SQLite or PostgreSQL you would need to list every column explicitly:
>
> `SELECT building_id, household_id, district_id, vdcmun_id FROM id_map` (tedious for wide tables)
>
> DuckDB's `EXCLUDE` is a quality-of-life addition for data scientists working with tables that have many columns.

**Code Task 4.1.4.3**: Import Gorkha data (district 4) from the `id_map.csv` into a Pandas DataFrame. Select `building_id` separately, then select all other columns excluding `building_id`. Set `building_id` as the index. Store in `df_gorkha_duckdb`.


In [ ]:
csv_path = './data/id_map.csv'

query = f"""
    SELECT DISTINCT building_id,
           * EXCLUDE (building_id)
    FROM '{csv_path}'
    WHERE district_id = 4
"""

# Convert to DataFrame and set index
df_gorkha_duckdb = duckdb.sql(query).df().set_index("building_id")

print(f"DataFrame shape: {df_gorkha_duckdb.shape}")
print(df_gorkha_duckdb.head())

ÃƒÂ¢Ã…â€œÃ¢â‚¬Â¦ **You may now attempt Multiple Choice Question 4.1.4.1 & Multiple Choice Question 4.1.4.2**

---

## Part 11: Performance Comparison ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â SQLite vs DuckDB

One of the main arguments for using DuckDB in data science workflows is speed on analytical queries. Let us measure this directly by running the same aggregation query in both databases and timing the result.

> ÃƒÂ°Ã…Â¸Ã¢â‚¬ÂÃ‚Â **What to look for:** we will run a `GROUP BY foundation_type` query with `COUNT(*)` and `AVG(age_building)` ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a representative analytical query that a data scientist runs frequently during EDA. We time each engine using `time.time()`.

> ÃƒÂ¢Ã…Â¡Ã‚Â ÃƒÂ¯Ã‚Â¸Ã‚Â **Important caveats about this benchmark:**
> 1. Absolute times depend on your hardware, file system, and Python overhead ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â your numbers will differ from the lesson solution
> 2. On a small dataset (234,835 rows) the difference may appear modest
> 3. On multi-million-row datasets, DuckDB's column-based vectorization produces dramatically larger speedups
> 4. The comparison is not perfectly fair: SQLite reads from a pre-indexed `.sqlite` file while DuckDB reads raw CSV ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â yet DuckDB is still comparable or faster

The goal here is to observe the *direction* of the difference and understand *why* it exists from the architectural principles in Part 6.

**Code Task 4.1.5.1**: Time a `GROUP BY` aggregation query in both SQLite (reading from `nepal.sqlite`) and DuckDB (reading from `building_structure.csv`). Use `time.time()` to capture start and end times. Print both results and the time taken.


In [ ]:
# 1. SQLite timing - reading from SQLite database
start = time.time()
conn = sqlite3.connect('./nepal.sqlite')
df_sqlite_agg = pd.read_sql_query("""
    SELECT foundation_type, COUNT(*) as count, AVG(age_building) as avg_age
    FROM building_structure
    GROUP BY foundation_type
""", conn)
conn.close()
sqlite_time = time.time() - start

# 2. DuckDB timing - reading from CSV file
start = time.time()
df_duckdb_agg = duckdb.sql("""
    SELECT foundation_type, COUNT(*) as count, AVG(age_building) as avg_age
    FROM read_csv_auto('./data/building_structure.csv')
    GROUP BY foundation_type
""").fetchdf()
duckdb_time = time.time() - start

print(f"SQLite time: {sqlite_time:.4f} seconds")
print(f"DuckDB time: {duckdb_time:.4f} seconds")
if duckdb_time > 0:
    print(f"DuckDB is {sqlite_time/duckdb_time:.1f}x faster!")

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  **Reading the performance result:** even on our relatively small dataset, DuckDB is comparable to or faster than SQLite for aggregation queries. This may seem surprising ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â SQLite is reading from a pre-indexed binary database file while DuckDB is reading raw CSV. The reason DuckDB keeps pace (or wins) comes from the three mechanisms in Part 6: column-based memory reads, SIMD vectorization, and query optimization.

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **The architectural takeaway:**
> - **SQLite reads all 9 columns** per row to compute `AVG(age_building)` ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â even though it only needs `age_building` and `foundation_type`
> - **DuckDB reads only 2 columns** from the CSV ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the exact columns needed by the query
> - On a 234,835-row dataset, this is a 4.5ÃƒÆ’Ã¢â‚¬â€ reduction in memory bandwidth
> - On a 10M-row dataset, this becomes a 4.5ÃƒÆ’Ã¢â‚¬â€ reduction at much larger absolute scale ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ dramatic wall-clock speedup

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **For Lessons 2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5:** DuckDB's columnar architecture is why we use it for all modeling. The `GROUP BY` operations in L2 feature engineering and the multi-table analytical joins in L4 will all run through DuckDB, benefiting from these performance advantages.

ÃƒÂ¢Ã…â€œÃ¢â‚¬Â¦ **You may now attempt Multiple Choice Question 4.1.5.1**

---

## Part 12: The Wrangle Function ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Reusable Data Loading

For Lessons 2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5, we need to load the Nepal data repeatedly ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â once per notebook, with potentially different district filters. Instead of writing the same multi-table SQL query every time, we encapsulate it in a **wrangle module**.

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **The wrangle-module pattern** is a recurring idiom in this course. A wrangle function:
> 1. Takes configuration parameters (file paths, district filter)
> 2. Runs the full loading and cleaning pipeline internally
> 3. Returns a DataFrame that is immediately ready for machine learning
>
> Benefits: (1) no copy-paste across notebooks, (2) a single place to fix bugs, (3) downstream lessons can focus on modeling, not data plumbing.

**What `wrangle_nepal_data()` does step by step:**

| Step | Operation | Why |
|------|-----------|-----|
| 1 | Read `building_structure.csv` and `building_damage.csv` with DuckDB | Columnar read, fast aggregations |
| 2 | JOIN structure + damage + id_map on `building_id` | Combine features with damage grades and district info |
| 3 | Filter by `district_id` (optional, default = 4 for Gorkha) | Allow per-district analysis in each lesson |
| 4 | Create binary target `severe_damage` | Grade 4 or 5 ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ 1 (severe); Grades 1ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“3 ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ 0 (not severe) |
| 5 | Drop `damage_grade` column | Target is now binary; original grade is no longer needed |
| 6 | Set `building_id` as the DataFrame index | Clean pandas index; no duplicate ID columns |

**Code 4.1.6.1**: Create the `wrangle_nepal_data()` function


In [ ]:
def wrangle_nepal_data(csv_path="./data", district_id=None):
    """
    Retrieve and prepare Nepal earthquake data for binary classification.

    This function reads CSV files using DuckDB, joins building structure
    with building damage, and creates a binary target variable 'severe_damage' where:
    - 1 = Grade 4 or Grade 5 damage (severe)
    - 0 = Grade 1, 2, or 3 damage (not severe)

    Parameters:
    -----------
    csv_path : str
        Path to the data folder (default: './data')
    district_id : int, optional
        Filter by specific district (e.g., 4 for Gorkha).
        If None, returns all districts.

    Returns:
    --------
    pd.DataFrame
        DataFrame with features and binary target 'severe_damage'
    """
    # Build file paths
    structure_path = f"{csv_path}/building_structure.csv"
    damage_path = f"{csv_path}/building_damage.csv"

    # Build WHERE clause
    where_clause = "WHERE b.damage_grade IS NOT NULL"
    if district_id is not None:
        where_clause += f" AND i.district_id = {district_id}"

    # Always join with id_map to get district_id
    query = f"""
        SELECT DISTINCT
            s.building_id,
            i.district_id,
            s.age_building,
            s.plinth_area_sq_ft,
            s.height_ft_pre_eq,
            s.foundation_type,
            s.ground_floor_type,
            b.damage_grade
        FROM read_csv_auto('{structure_path}') s
        JOIN read_csv_auto('{damage_path}') b ON s.building_id = b.building_id
        JOIN read_csv_auto('./data/id_map.csv') i ON s.building_id = i.building_id
        {where_clause}
    """

    return (
        duckdb.sql(query)
        .df()
        .set_index("building_id")
        .assign(
            severe_damage=lambda x: (
                x["damage_grade"]
                .str.contains("Grade 4|Grade 5")
                .fillna(False)
                .astype(int)
            )
        )
        .drop(columns=["damage_grade"])
    )

> ÃƒÂ°Ã…Â¸Ã‚Â§Ã‚Â  **Two design decisions to notice in the function:**
>
> **1. The binary target creation:** `damage_grade.str.contains("Grade 4|Grade 5")` converts the raw ordinal grade (Grade 1ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5) into a binary label (0/1). This is a modeling choice: we treat Grade 4 and 5 together as "severe" because they require substantial structural intervention. Grade 1ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“3 buildings are classified as "not severe."
>
> Why binary instead of five ordinal classes? Binary classification is simpler to model, easier to evaluate (precision/recall, confusion matrix, ROC/AUC), and aligns with the practical decision a disaster response team faces: does this building need immediate major intervention or not?
>
> **2. The `.drop(columns=["damage_grade"])` step:** once we have created `severe_damage`, the original `damage_grade` column is removed. Keeping it would cause **target leakage** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the raw damage grade is almost perfectly correlated with our binary target (Grade 4 ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ severe=1, Grade 3 ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ severe=0). A model that has access to `damage_grade` would trivially predict `severe_damage` with near-100% accuracy, but it would learn nothing about *building features* that predict damage. Target leakage will be covered in depth in Lesson 2.

> ÃƒÂ¢Ã…Â¾Ã‚Â¡ÃƒÂ¯Ã‚Â¸Ã‚Â Now use the function to load the Gorkha data and verify the output shape and target rate.

**Code Task 4.1.6.2**: Use `wrangle_nepal_data()` to load data for **Gorkha** (district_id = 4). Store in `df_wrangled`. Print the shape, columns, and severe damage rate.


In [ ]:
# Gorkha is district_id 4
df_wrangled = wrangle_nepal_data('./data', district_id=4)
print(f"Shape: {df_wrangled.shape}")
print(f"Columns: {list(df_wrangled.columns)}")
print(f"Severe damage rate: {df_wrangled['severe_damage'].mean():.2%}")

**Solution 4.1.6.2**

In [ ]:
# Gorkha is district_id 4
df_wrangled = wrangle_nepal_data('./data', district_id=4)
print(f"Shape: {df_wrangled.shape}")
print(f"Columns: {list(df_wrangled.columns)}")
print(f"Severe damage rate: {df_wrangled['severe_damage'].mean():.2%}")

ÃƒÂ¢Ã…â€œÃ¢â‚¬Â¦ **You may now attempt Multiple Choice Question 4.1.6.1**

> ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  **Reading the output:** ~70,836 rows (Gorkha's buildings), 7 columns (6 features + the binary target `severe_damage`), and a severe damage rate of approximately 64%. That rate means almost two-thirds of Gorkha's surveyed buildings suffered Grade 4 or Grade 5 damage ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â a striking figure that will shape every modeling decision in Lessons 2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5. A naÃƒÆ’Ã‚Â¯ve model that always predicts "severe" would be correct 64% of the time ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â this is our **baseline accuracy** to beat in L2.

---

Now let us save the function to a file so future lessons can import it with a single line.

> ÃƒÂ°Ã…Â¸Ã¢â‚¬â„¢Ã‚Â¡ **The `%%writefile` IPython magic** writes the contents of the current cell to a Python file on disk. Running the cell creates (or overwrites) `duckdb_wrangle.py` in the current directory. Lessons 2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“5 will import the function with:
> ```python
> from duckdb_wrangle import wrangle_nepal_data
> ```

> ÃƒÂ¢Ã…Â¡Ã‚Â ÃƒÂ¯Ã‚Â¸Ã‚Â **Important:** `%%writefile` must be the **very first line** of the code cell ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â no spaces, no comments before it. If you see `Overwriting duckdb_wrangle.py`, it means the file already existed and was replaced. If you see `Writing duckdb_wrangle.py`, the file was created fresh. Both are correct outcomes.

**Code 4.1.6.3**: Save the `wrangle_nepal_data()` function to `duckdb_wrangle.py` for use in subsequent lessons


In [ ]:
%%writefile duckdb_wrangle.py
"""
duckdb_wrangle.py

Data wrangling module for Project 4: Nepal Earthquake Damage Prediction.
This module provides functions to query CSV data using DuckDB for machine learning classification tasks.

Usage:
    from duckdb_wrangle import wrangle_nepal_data

    # All data (default)
    df = wrangle_nepal_data("./data")

    # Specific district (e.g., Gorkha = district_id 4)
    df_gorkha = wrangle_nepal_data("./data", district_id=4)
"""

import duckdb
import pandas as pd


def wrangle_nepal_data(csv_path="./data", district_id=None):
    """
    Retrieve and prepare Nepal earthquake data for binary classification.

    This function reads CSV files using DuckDB and joins building structure
    with building damage data. The damage_grade column is kept as-is for use
    in creating the target variable in Lesson 2.

    Parameters:
    -----------
    csv_path : str
        Path to the data folder (default: "./data")
    district_id : int, optional
        Filter by specific district (e.g., 4 for Gorkha).
        If None, returns all districts.

    Returns:
    --------
    pd.DataFrame
        DataFrame with features and damage_grade column
    """
    # Build file paths
    structure_path = f"{csv_path}/building_structure.csv"
    damage_path = f"{csv_path}/building_damage.csv"

    # Build WHERE clause
    where_clause = "WHERE b.damage_grade IS NOT NULL"
    if district_id is not None:
        where_clause += f" AND i.district_id = {district_id}"

    # Always join with id_map to get district_id
    query = f"""
        SELECT DISTINCT
            s.building_id,
            i.district_id,
            s.age_building,
            s.plinth_area_sq_ft,
            s.height_ft_pre_eq,
            s.foundation_type,
            s.ground_floor_type,
            b.damage_grade
        FROM read_csv_auto('{structure_path}') s
        JOIN read_csv_auto('{damage_path}') b ON s.building_id = b.building_id
        JOIN read_csv_auto('./data/id_map.csv') i ON s.building_id = i.building_id
        {where_clause}
    """

    return (
        duckdb.sql(query)
        .df()
        .set_index("building_id")
        .assign(
            severe_damage=lambda x: (
                x["damage_grade"]
                .str.contains("Grade 4|Grade 5")
                .fillna(False)
                .astype(int)
            )
        )
        .drop(columns=["damage_grade"])
    )

## Summary and Discussion

This lesson introduced two complementary tools for working with relational data and the foundational concepts that connect them.

### What you built and learned

| Concept / skill | Key takeaway |
|-----------------|-------------|
| **Database normalization** | Eliminates update, insert, and delete anomalies; each fact stored exactly once |
| **Data integrity constraints** | PK, FK, NOT NULL, UNIQUE, CHECK enforce correctness before Python even runs |
| **SQL execution order** | FROM ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ WHERE ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ GROUP BY ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ HAVING ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ SELECT ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ ORDER BY ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ LIMIT; not the write order |
| **INNER JOIN** | Intersection ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â only matched rows; silently drops unmatched rows; always verify row counts |
| **LEFT JOIN** | Left-inclusive ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â all left rows, NULLs where no match; reveals missing data |
| **OLTP vs OLAP** | SQLite = row-based, ACID, transaction-optimized; DuckDB = column-based, analytics-optimized |
| **DuckDB advantages** | Direct CSV queries, `EXCLUDE` keyword, SIMD vectorization, 10ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“100ÃƒÆ’Ã¢â‚¬â€ faster for GROUP BY |
| **`wrangle_nepal_data()`** | Encapsulates JOIN + filter + binary-target creation in one reusable function |
| **Severe damage rate** | ~64% of Gorkha's buildings suffered severe damage ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the base rate our models must improve on |
| **Binary target** | Grade 4 or 5 ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ severe=1; Grade 1ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“3 ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ severe=0; `damage_grade` dropped to prevent leakage |

### When to use each tool

| Situation | Recommendation | Why |
|-----------|---------------|-----|
| Mobile app with user data | SQLite | Embedded, ACID transactions, small footprint |
| Analyzing CSV or Parquet files | DuckDB | Direct file query, no import step |
| GROUP BY queries over large datasets | DuckDB | Columnar storage + vectorization |
| Need ACID transaction guarantees | SQLite | DuckDB is read-optimized, not write-optimized |
| Quick analytical exploration | DuckDB | Query CSV directly; no schema setup |
| Embedded app with zero dependencies | SQLite | In Python standard library |

### Key terms to know

| Term | One-line definition |
|------|-------------------|
| **Normalization** | Organizing a database to eliminate redundancy |
| **Primary key** | Unique, non-null row identifier |
| **Foreign key** | Column that references a PK in another table |
| **INNER JOIN** | Returns only matched rows from both tables |
| **LEFT JOIN** | Returns all left rows + matched right rows (NULL for no match) |
| **OLTP** | Row-based, transaction-optimized database (SQLite, MySQL) |
| **OLAP** | Column-based, analytics-optimized database (DuckDB, BigQuery) |
| **Target leakage** | Using information in features that is directly derived from the target ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â inflates accuracy |
| **Severe damage** | Damage Grade 4 or 5 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â binary target for classification in L2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“L5 |

### Discussion Questions

1. Why might you still use SQLite even though DuckDB is faster for analytics? What guarantees does SQLite provide that DuckDB does not?
2. When you JOIN the four Nepal tables, what relationships are you relying on? Why can't you simply use `DISTINCT` on `building_id` from `id_map` without joining to `building_structure`?
3. The severe damage rate in Gorkha is ~64%. What does this imply about the earthquake's impact, and what does it mean for a machine learning model that naÃƒÆ’Ã‚Â¯vely predicts "severe" for every building?
4. The `wrangle_nepal_data()` function creates a binary target by collapsing five damage grades into two. What information is lost in this collapse? Under what circumstances might the five-grade ordinal target be a better modeling choice?
5. What other file formats can DuckDB query directly? When would you prefer Parquet over CSV?
6. The three database anomalies (update, insert, delete) motivated the four-table design. Can you identify a specific scenario in the Nepal data where each anomaly would have occurred if all data were in a single denormalized table?

### Next Steps

In Lesson 2, you will:
- Import data using `from duckdb_wrangle import wrangle_nepal_data`
- Build your first **logistic regression** classification model on the Nepal damage data
- Learn about **baseline accuracy**, **precision**, **recall**, **confusion matrices**, and **ROC/AUC**
- Understand why **data leakage** is the silent killer of production models ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â and encounter three specific examples from the Nepal data
- Evaluate whether a model that achieves 70% accuracy is actually good given a 64% base rate

> ÃƒÂ¢Ã…Â¾Ã‚Â¡ÃƒÂ¯Ã‚Â¸Ã‚Â Lesson 2 starts from the output of the function you just saved ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â the cleaned Gorkha DataFrame with `severe_damage` as the binary target and six building-feature columns as predictors.
